# Lab04.5 — Vision-Controlled Zybo LEDs

**Zybo Legacy · PYNQ 3.0.1 · OpenCV 4.5.4 · Logitech C270**

**Objective:** use the position of the largest colored region detected by the webcam to control four on-board LEDs. The HSV + RGB segmentation is the same as in the verified Lab04.4.

| Object position | LED | Bit pattern |
|---|---|---|
| LEFT | LD0 | `0001` |
| CENTER | LD1 | `0010` |
| RIGHT | LD2 | `0100` |
| LOST (no matching region) | LD3 | `1000` |

LEDs are switched off when the test finishes, including after a normal Jupyter interrupt. Use only one notebook accessing the camera at a time.

## 1. Camera, segmentation, and position functions

Run this cell first. Select the desired color using `TARGET_COLOR`. This cell does not open the camera or modify GPIO.

In [ ]:
import cv2
import numpy as np
import time
from contextlib import contextmanager
from IPython.display import display, Image, clear_output

CAMERA_DEVICE = "/dev/video0"
TARGET_COLOR = "GREEN"    # RED, GREEN, or BLUE
FRAME_WIDTH, FRAME_HEIGHT = 640, 480
WARMUP_FRAMES = 30
MIN_AREA = 150

COLORS = {
    "RED": {"ranges": [((0, 100, 50), (10, 255, 255)),
                       ((170, 100, 50), (179, 255, 255))],
            "margin": 35, "box": (0, 0, 255)},
    "GREEN": {"ranges": [((35, 45, 35), (90, 255, 255))],
              "margin": 10, "box": (0, 255, 0)},
    "BLUE": {"ranges": [((95, 80, 50), (130, 255, 255))],
             "margin": 35, "box": (255, 0, 0)}
}

@contextmanager
def camera_session():
    cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
    try:
        if not cap.isOpened():
            raise RuntimeError("Cannot open camera: " + CAMERA_DEVICE)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)
        for _ in range(WARMUP_FRAMES):
            ok, frame = cap.read()
            if not ok or frame is None:
                raise RuntimeError("Camera warm-up failed.")
            time.sleep(0.05)
        yield cap
    finally:
        cap.release()
        print("Camera released.")

def show_bgr(frame, replace=False):
    ok, jpg = cv2.imencode(".jpg", frame)
    if not ok:
        raise RuntimeError("JPEG encoding failed.")
    if replace:
        clear_output(wait=True)
    display(Image(data=jpg.tobytes()))

def detect_position(frame, color_name):
    if color_name not in COLORS:
        raise ValueError("Color must be RED, GREEN, or BLUE")
    cfg = COLORS[color_name]
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    b, g, r = [c.astype(np.int16) for c in cv2.split(frame)]
    hsv_mask = np.zeros(hsv.shape[:2], np.uint8)
    for low, high in cfg["ranges"]:
        hsv_mask = cv2.bitwise_or(hsv_mask, cv2.inRange(hsv, low, high))
    m = cfg["margin"]
    if color_name == "RED":
        dominant = (r > g + m) & (r > b + m) & (r > 80)
    elif color_name == "GREEN":
        dominant = (g > r + m) & (g > b + m) & (g > 50)
    else:
        dominant = (b > r + m) & (b > g + m) & (b > 80)
    mask = cv2.bitwise_and(hsv_mask, dominant.astype(np.uint8) * 255)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    mask = cv2.morphologyEx(
        mask, cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7)))
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    candidates = [c for c in contours if cv2.contourArea(c) >= MIN_AREA]
    result = frame.copy()
    h, w = frame.shape[:2]
    cv2.line(result, (w // 3, 0), (w // 3, h - 1), (160, 160, 160), 1)
    cv2.line(result, (2 * w // 3, 0), (2 * w // 3, h - 1), (160, 160, 160), 1)
    state, centroid = "LOST", None
    if candidates:
        cnt = max(candidates, key=cv2.contourArea)
        x, y, bw, bh = cv2.boundingRect(cnt)
        M = cv2.moments(cnt)
        if M["m00"]:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
        else:
            cx, cy = x + bw // 2, y + bh // 2
        centroid = (cx, cy)
        state = "LEFT" if cx < w / 3 else ("RIGHT" if cx >= 2*w / 3 else "CENTER")
        cv2.rectangle(result, (x, y), (x + bw, y + bh), cfg["box"], 2)
        cv2.circle(result, centroid, 5, cfg["box"], -1)
    cv2.putText(result, "{}: {}".format(color_name, state),
                (12, h-16), cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (255, 255, 255), 2)
    return state, centroid, result

print("OpenCV:", cv2.__version__, "| Target:", TARGET_COLOR)

## 2. Initialize the Zybo LEDs and verify each pattern

This uses the same **`base.bit` / `leds_gpio.channel1`** interface tested in Lab01. Running the initialization reloads the base overlay. **Do not run other overlay notebooks concurrently.** Each LED is lit briefly and all LEDs are turned off afterward.

In [ ]:
from pynq import Overlay

ol = Overlay("base.bit")
leds = ol.leds_gpio.channel1
leds.setdirection("out")
leds.setlength(4)

LED_PATTERN = {"LEFT": 0b0001, "CENTER": 0b0010,
               "RIGHT": 0b0100, "LOST": 0b1000}

def leds_off():
    leds.write(0, 0b1111)

def set_position_led(state):
    leds.write(LED_PATTERN[state], 0b1111)

try:
    for state in ("LEFT", "CENTER", "RIGHT", "LOST"):
        print("LED test:", state)
        set_position_led(state)
        time.sleep(0.7)
finally:
    leds_off()
    print("All LEDs off.")

## 3. Single-frame vision-to-LED test

Hold a **green marker** in view and execute this cell. The matching LED lights briefly, then switches off. Repeat with the marker in LEFT, CENTER and RIGHT. Remove the marker and check LD3. The LED is automatically reset and the camera is released.

In [ ]:
try:
    with camera_session() as cap:
        ok, frame = cap.read()
        if not ok or frame is None:
            raise RuntimeError("No camera frame captured.")
        state, centroid, annotated = detect_position(frame, TARGET_COLOR)
        set_position_led(state)
        print("Position:", state, "| Centroid:", centroid)
        show_bgr(annotated)
        time.sleep(1.5)  # Keep the LED on long enough for visual inspection.
finally:
    leds_off()
    print("All LEDs off.")

## 4. Vision-controlled LEDs in successive frames

Move a marker from left to right and remove it once to test **LOST**. The exercise stops after **60 frames or 12 seconds**, whichever comes first. The LED updates for every successfully processed frame; Jupyter displays only every fifth frame to limit overhead. Camera and LEDs are released/reset on exceptions and ordinary interrupts.

**Note:** time limits are checked between USB reads. A blocked camera-driver read can delay a timeout.

In [ ]:
MAX_FRAMES = 60
MAX_SECONDS = 12
DISPLAY_EVERY = 5
processed = 0
last_state = "LOST"
last_image = None

try:
    leds_off()
    with camera_session() as cap:
        start = time.monotonic()
        for i in range(MAX_FRAMES):
            if time.monotonic() - start >= MAX_SECONDS:
                break
            ok, frame = cap.read()
            if not ok or frame is None:
                print("WARNING: Camera frame read failed.")
                break
            last_state, centroid, last_image = detect_position(frame, TARGET_COLOR)
            set_position_led(last_state)
            processed += 1
            if i % DISPLAY_EVERY == 0:
                show_bgr(cv2.resize(last_image, (480, 360)), replace=True)
finally:
    leds_off()
    print("All LEDs off.")

print("Processed frames:", processed, "| Last state:", last_state)
if last_image is not None:
    show_bgr(cv2.resize(last_image, (480, 360)))

## Student tasks and acceptance criteria

1. Confirm the manual GPIO test: LEFT → LD0, CENTER → LD1, RIGHT → LD2, LOST → LD3.
2. With `TARGET_COLOR = "GREEN"`, run the single-frame test in all four states.
3. During the 60-frame session move the object across the image and watch the corresponding LEDs.
4. Remove the object: LD3 should light while tracking is active. Check that all LEDs switch **off at the end**.
5. Change the target to BLUE or RED and repeat the test. Explain one case when the largest **colored region** is not equivalent to an entire object.

**Expected behavior:** only one LED is active at a time; no matching region means LD3; the camera is released and all LEDs are off after the test. Use **Kernel → Interrupt** to stop early.

**Limitation:** this is region tracking, not identity tracking. A different object of the same color may be selected if its colored region becomes larger.